# Full Dataset Training (Google Drive)
Mount Drive, train from full JSONL, and save model artifacts back to Drive.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/sentiment-engine'
DATASET_FILE = os.path.join(BASE_DIR, 'datasets/train_qwen_28_full.jsonl')
OUTPUT_DIR = os.path.join(BASE_DIR, 'qwen_sentiment_finetuned_full')

os.makedirs(BASE_DIR, exist_ok=True)
os.makedirs(os.path.dirname(DATASET_FILE), exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('Dataset:', DATASET_FILE)
print('Output:', OUTPUT_DIR)

In [ ]:
!pip -q install unsloth trl datasets accelerate bitsandbytes

In [ ]:
import os
import torch
from datasets import load_dataset
from unsloth import FastLanguageModel
from trl.trainer.sft_trainer import SFTTrainer
from trl.trainer.sft_config import SFTConfig

MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = False

BASE_DIR = '/content/drive/MyDrive/sentiment-engine'
DATASET_FILE = os.path.join(BASE_DIR, 'datasets/train_qwen_28_full.jsonl')
OUTPUT_DIR = os.path.join(BASE_DIR, 'qwen_sentiment_finetuned_full')

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(f'Missing dataset file: {DATASET_FILE}')

print('Loading Qwen 2.5 1.5B model...')
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name='Qwen/Qwen2.5-1.5B-Instruct',
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj',
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

alpaca_prompt = '''Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}'''

EOS_TOKEN = tokenizer.eos_token

def formatting_prompts_func(examples):
    instructions = examples['instruction']
    inputs = examples['input']
    outputs = examples['output']
    texts = []
    for instruction, inp, output in zip(instructions, inputs, outputs):
        texts.append(alpaca_prompt.format(instruction, inp, output) + EOS_TOKEN)
    return {'text': texts}

print(f'Loading dataset: {DATASET_FILE}')
dataset = load_dataset('json', data_files=DATASET_FILE, split='train')
dataset = dataset.map(formatting_prompts_func, batched=True)

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    max_length=MAX_SEQ_LENGTH,
    dataset_text_field='text',
    dataset_num_proc=2,
    packing=False,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    warmup_steps=5,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    save_strategy='steps',
    save_steps=200,
    save_total_limit=3,
    optim='adamw_8bit',
    weight_decay=0.01,
    lr_scheduler_type='linear',
    seed=3407,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    args=sft_config,
)

print('Starting training...')
has_checkpoint = os.path.isdir(OUTPUT_DIR) and any(
    name.startswith('checkpoint-') for name in os.listdir(OUTPUT_DIR)
)
trainer.train(resume_from_checkpoint=True if has_checkpoint else None)

print('Saving adapters + tokenizer to Drive...')
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Done. Saved to: {OUTPUT_DIR}')